# Explore AI Agent development

## Create an Azure AI Foundry project and agent

### Provision a resource group

In [1]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
LOCATION = os.getenv("LOCATION")

# Authenticate
credential = DefaultAzureCredential()

# Create resource group
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
print(f"Creating resource group {RESOURCE_GROUP}...")
resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP,
    {"location": LOCATION}
)
print(f"Resource group {RESOURCE_GROUP} created successfully.")

Creating resource group rg-agent-workshop...
Resource group rg-agent-workshop created successfully.


### Provision an Azure AI Foundry resource:

In [2]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
AI_FOUNDRY_RESOURCE_NAME = os.getenv("AI_FOUNDRY_RESOURCE_NAME")
AI_FOUNDRY_PROJECT_NAME = os.getenv("AI_FOUNDRY_PROJECT_NAME")
LOCATION = os.getenv("LOCATION")

# Authenticate
credential = DefaultAzureCredential()

cs_client = CognitiveServicesManagementClient(
    credential,
    SUBSCRIPTION_ID,
    api_version="2025-04-01-preview")

print(f"Creating Azure AI Foundry resource {AI_FOUNDRY_RESOURCE_NAME}...")
cs_account = cs_client.accounts.begin_create(
    resource_group_name=RESOURCE_GROUP,
    account_name=AI_FOUNDRY_RESOURCE_NAME,
    account={
        "kind": "AIServices",
        "location": LOCATION,
        "sku": {"name": "S0"},
        "identity": {"type": "SystemAssigned"},
        "properties": {
            "allowProjectManagement": True,
            "customSubDomainName": AI_FOUNDRY_RESOURCE_NAME
        }
    }
).result()
print(f"Azure AI Foundry resource {AI_FOUNDRY_RESOURCE_NAME} created successfully.")


Creating Azure AI Foundry resource agent-foundry-lab1...
Azure AI Foundry resource agent-foundry-lab1 created successfully.


View project settings

In [9]:
project = cs_client.projects.get(
    resource_group_name=RESOURCE_GROUP,
    account_name=AI_FOUNDRY_RESOURCE_NAME,
    project_name=AI_FOUNDRY_PROJECT_NAME
)

print(project) 

ResourceNotFoundError: (ResourceNotFound) The Resource 'Microsoft.CognitiveServices/accounts/agent-foundry-lab1/projects/agent-foundry-pj' under resource group 'rg-agent-workshop' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix
Code: ResourceNotFound
Message: The Resource 'Microsoft.CognitiveServices/accounts/agent-foundry-lab1/projects/agent-foundry-pj' under resource group 'rg-agent-workshop' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix

In [ ]:
az cognitiveservices account show --name agent-foundry-lab1 --resource-group rg-agent-workshop

Retrieve endpoint and API key for the Azure AI Foundry resource:

In [ ]:
# Retrieve endpoint
az cognitiveservices account show \
  --resource-group rg-agent-workshop \
  --name agent-foundry-lab1 \
  --query properties.endpoint -o tsv > endpoint.txt

In [ ]:

# Retrieve API key
az cognitiveservices account keys list \
  --resource-group rg-agent-workshop \
  --name agent-foundry-lab1 \
  --query key1 -o tsv > apikey.txt

### Provision a model deployment:

List available models:

In [ ]:
az cognitiveservices account list-models \
  -n agent-foundry-lab1 \
  -g rg-agent-workshop | \
  jq '.[] | { 
    name: .name, 
    format: .format, 
    version: .version, 
    sku: .skus[0].name, 
    capacity: .skus[0].capacity.default 
  }'

In [ ]:
{
  "name": "gpt-5-mini",
  "format": "OpenAI",
  "version": "2025-08-07",
  "sku": "GlobalStandard",
  "capacity": 10
}

Deploy the `gpt-5-mini` (CLI Azure) model to the Azure AI Foundry resource.
* Este comando usa el recurso Foundry directamente
* `sku-capacity` puede ajustarse según el laboratorio (ej. 50 para 50K TPM)

In [ ]:
az cognitiveservices account deployment create \
  --name agent-foundry-lab1 \
  --resource-group rg-agent-workshop \
  --deployment-name gpt5mini-deployment \
  --model-name gpt-5-mini \
  --model-version "2025-08-07" \
  --model-format OpenAI \
  --sku-name GlobalStandard \
  --sku-capacity 50

### Create the Azure AI Foundry project and agent:

In [1]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
AI_FOUNDRY_RESOURCE_NAME = os.getenv("AI_FOUNDRY_RESOURCE_NAME")
PROJECT_NAME = os.getenv("PROJECT_NAME")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AGENT_NAME = os.getenv("AGENT_NAME")
VECTOR_STORE_NAME = os.getenv("VECTOR_STORE_NAME")
DOC_PATH = "Expenses_Policy.docx"
project_endpoint = os.getenv("PROJECT_ENDPOINT")

In [2]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Initialize project client
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential
)


In [4]:
import requests
token = credential.get_token("https://ai.azure.com/.default").token

# Create vector store and upload document
response = requests.post(
    f"{project_endpoint}/vector_stores?api-version=v1",
    headers={"Authorization": f"Bearer {token}"},
    json={
        "name": VECTOR_STORE_NAME,
        "file_ids": [],  # Se puede llenar luego
        "configuration": {},
        "metadata": {"created_by": "Luis"}
    }
)

print(response.status_code, response.json())

400 {'error': {'message': "Missing required parameter: 'configuration.data_sources'.", 'type': 'invalid_request_error', 'param': 'configuration.data_sources', 'code': 'missing_required_parameter'}}


In [ ]:
# Create agent
from azure.ai.agents import AgentsClient, CodeInterpreterToolDefinition

agents_client = AgentsClient(
    endpoint=project_endpoint,
    credential=credential
)

print(f"Creating agent: {AGENT_NAME}")
agent = agents_client.create_agent(
    model=MODEL_DEPLOYMENT_NAME,
    name=AGENT_NAME,
    instructions="""You are an AI assistant for corporate expenses.
You answer questions about expenses based on the expenses policy data.
If a user wants to submit an expense claim, you get their email address, a description of the claim, and the amount to be claimed and write the claim details to a text file that the user can download.""",
    tools=[CodeInterpreterToolDefinition()],
    tool_resources={"vector_store": {"ids": [vector_store.id]}}
)


In [ ]:
# Chat with the agent
response = agent.chat(messages=[
    {"role": "user", "content": "Can I claim a taxi ride from the airport to the hotel?"}
])
print(response.choices[0].message["content"])
